In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv


## Imports

In [3]:
import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
import wandb
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)
torch.manual_seed(42) 


Device available:  cuda


## Data Import / Data Split

In [4]:
df = pd.read_csv("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv")
df.groupby('emotion').size()

emotion
0    3995
1     436
2    4097
3    7215
4    4830
5    3171
6    4965
dtype: int64

In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,stratify=df['emotion'])


## Dataset Class Initialization

In [45]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        # Parse all pixel strings once, upfront
        pixels = np.stack([
            np.array(p.split(), dtype=np.float32).reshape(1, 48, 48) / 255.0
            for p in df['pixels'].values
        ])
        self.pixels = torch.tensor(pixels)  # shape: (N, 1, 48, 48)
        self.labels = df['emotion'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.pixels[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
## Since Data is small augmentation helps model training

transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),])

In [46]:
train_dataset = CustomDataset(train_df,transform=None)
val_dataset = CustomDataset(val_df,transform=None)

In [6]:
LR = 0.001
BATCH_SIZE = 64

In [32]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True)

## Model Initialization

#### 48x48 -> maxpool(2,2) -> 24x24 ->maxpool(2,2) -> 12x12 -> fc

> Initial Model






In [ ]:
class Net(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # (32, 48, 48)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (64, 24, 24)
    self.conv3 = nn.Conv2d(64,128, kernel_size=3, padding=1)  # (128, 12,12)
    self.pool = nn.MaxPool2d(2,2)
    self.relu = nn.ReLU()

    self.flatten = nn.Flatten()
    self.fc = nn.Linear(128*12*12,256)
    self.output = nn.Linear(256,7)

  def forward(self,x):
    x = self.conv1(x)
    x = self.pool(x)
    x = self.relu(x)

    x = self.conv2(x)
    x = self.pool(x)
    x = self.relu(x)

    x = self.conv3(x)
    x = self.relu(x)

    x = self.flatten(x)
    x = self.fc(x)
    x = self.relu(x)

    x = self.output(x)

    return x

model = Net().to(device)

In [34]:
from torchsummary import summary
summary(model, (1, 48, 48))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 48, 48]             320
         MaxPool2d-2           [-1, 32, 24, 24]               0
              ReLU-3           [-1, 32, 24, 24]               0
            Conv2d-4           [-1, 64, 24, 24]          18,496
         MaxPool2d-5           [-1, 64, 12, 12]               0
              ReLU-6           [-1, 64, 12, 12]               0
            Conv2d-7          [-1, 128, 12, 12]          73,856
              ReLU-8          [-1, 128, 12, 12]               0
           Flatten-9                [-1, 18432]               0
           Linear-10                  [-1, 256]       4,718,848
             ReLU-11                  [-1, 256]               0
           Linear-12                    [-1, 7]           1,799
Total params: 4,813,319
Trainable params: 4,813,319
Non-trainable params: 0
---------------------------

In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=LR)

In [ ]:
import wandb
!wandb login

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


#### Quick check to see if model is behaving correctly


*   Should overfit on small batch
*   Forward pass should return equal probs


#### Helper functions for sanity check / model training and logging

#### sanity_check()

In [9]:
import copy

def sanity_check(model, train_loader, model_name, LR, device, optimizer, criterion, n_classes=7, overfit_batch_size=4):

    run = wandb.init(
        entity="ldavi22-free-university-of-tbilisi-",
        project="Emotion Recognition",
        name=f"Sanity check {model_name}",
        config={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 10,
        },
    )

    # Work on copies so the real model/optimizer aren't mutated by this check
    model_copy = copy.deepcopy(model).to(device)
    optimizer_copy = type(optimizer)(model_copy.parameters(), lr=LR)

    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)

    # --- Loss @ init check ---
    model_copy.eval()

    with torch.no_grad():
        output = model_copy(images)
        loss_init = criterion(output, labels)
        probs = torch.softmax(output, dim=1)

        expected_loss = -torch.log(torch.tensor(1.0 / n_classes)).item()

        wandb.log({
            "init_check/loss": loss_init.item(),
            "init_check/expected_loss": expected_loss,
            "init_check/sample_prob": probs[0][0].item(),
            "init_check/prob_sum": probs[0].sum().item(),
        })

        print(f"Loss @ init: {loss_init.item():.4f} | Expected (-log(1/{n_classes})): {expected_loss:.4f}")

    # --- Overfit a tiny batch ---
    model_copy.train()

    tiny_images = images[:overfit_batch_size]
    tiny_labels = labels[:overfit_batch_size]

    for step in range(200):
        output = model_copy(tiny_images)
        loss = criterion(output, tiny_labels)

        optimizer_copy.zero_grad()
        loss.backward()
        optimizer_copy.step()

        acc = (output.argmax(dim=1) == tiny_labels).float().mean()

        wandb.log({
            "overfit/loss": loss.item(),
            "overfit/acc": acc.item(),
            "step": step
        })

    wandb.finish()

In [36]:
sanity_check(model,train_loader,"CustomCNN with normalized input",LR,device,optimizer,criterion)

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▂▄█▄▄▅▁▄▇█▅▄▆▆█▁▅▇▅▄▆▄▆▃▆▅▃▃▄▄▃▅▅▇▁▄▆▂▄
val/acc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,▇█▃▅▇▅▆▅█▇▁▅▄█▆▆▅▃▅▄▂▄▂▇▇▃▄▄▄▂▅▄▄▄▃▄▆▄▅
epoch,38
lr,0.001
train/acc,14.86916
train/loss,4.31938
val/acc,13.98467


Loss @ init: 1.9459 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁███████████████████████████████████████
overfit/loss,█▆▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇██
init_check/expected_loss,1.94591
init_check/loss,1.94593
init_check/prob_sum,1
init_check/sample_prob,0.1314


#### Training model on 50 epochs

In [41]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=LR)

#### train_model / log

In [10]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="run",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra=None, checkpoint_path="best_model.pt"):

    config = {"epochs": epochs}
    if config_extra:
        config.update(config_extra)

    run = wandb.init(entity=entity, project=project, name=run_name, config=config)
    wandb.watch(model, criterion, log="all", log_freq=10)

    n_train_samples = len(train_loader.dataset)
    n_val_samples = len(val_loader.dataset)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss_train = 0
        total_acc_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            output = model(inputs)
            loss = criterion(output, labels)
            total_loss_train += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_acc_train += (torch.argmax(output, axis=1) == labels).sum().item()

        model.eval()
        total_loss_val = 0
        total_acc_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                output = model(inputs)
                loss = criterion(output, labels)
                total_loss_val += loss.item()

                total_acc_val += (torch.argmax(output, axis=1) == labels).sum().item()

        avg_train_loss = total_loss_train / len(train_loader)
        avg_val_loss = total_loss_val / len(val_loader)
        avg_train_acc = (total_acc_train / n_train_samples) * 100
        avg_val_acc = (total_acc_val / n_val_samples) * 100

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        if scheduler is not None:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": avg_train_acc,
            "val/loss": avg_val_loss,
            "val/acc": avg_val_acc,
            "lr": optimizer.param_groups[0]["lr"],
            "epoch": epoch
        })

    wandb.finish()
    return history

#### Added Image Normalization 


In [ ]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="CustomCNN v1 rerun with norm1",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 50,
        }, checkpoint_path="best_model.pt")

#### Added BatchNorm

In [57]:
class NetWithBatchNorm(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # (32, 48, 48)
    self.bn1   = nn.BatchNorm2d(32)

    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (64, 24, 24)
    self.bn2   = nn.BatchNorm2d(64)

    self.conv3 = nn.Conv2d(64,128, kernel_size=3, padding=1)  # (128, 12,12)
    self.bn3   = nn.BatchNorm2d(128)

    self.pool = nn.MaxPool2d(2,2)
    self.relu = nn.ReLU()

    self.flatten = nn.Flatten()
    self.fc = nn.Linear(128*12*12,256)
    self.bn_fc = nn.BatchNorm1d(256)

    self.output = nn.Linear(256,7)

  def forward(self,x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.pool(x)
    x = self.relu(x)

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.pool(x)
    x = self.relu(x)

    x = self.conv3(x)
    x = self.bn3(x)
    x = self.relu(x)

    x = self.flatten(x)
    x = self.fc(x)
    x = self.bn_fc(x)
    x = self.relu(x)
    x = self.output(x)

    return x

model1 = NetWithBatchNorm().to(device)



In [58]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=1e-4)

train_loader = DataLoader(train_dataset,batch_size=4*BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4*BATCH_SIZE,shuffle=True)

In [56]:
sanity_check(model1,train_loader,"Sanity check CustomCNN with BatchNorm",1e-4,device,optimizer,criterion)

Loss @ init: 1.9566 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁███████████████████████████████████████
overfit/loss,█▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▁▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇██
init_check/expected_loss,1.94591
init_check/loss,1.95664
init_check/prob_sum,1
init_check/sample_prob,0.14731


In [ ]:
result = train_model(model1, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="CustomCNN v1 added batchnorm",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": 1e-4,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 50,
            "batch size" : 256,
        }, checkpoint_path="best_model.pt")

#### Removed BatchNorm after FC layer

In [67]:
class NetWithBatchNormv2(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # (32, 48, 48)
    self.bn1   = nn.BatchNorm2d(32)

    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (64, 24, 24)
    self.bn2   = nn.BatchNorm2d(64)

    self.conv3 = nn.Conv2d(64,128, kernel_size=3, padding=1)  # (128, 12,12)
    self.bn3   = nn.BatchNorm2d(128)

    self.pool = nn.MaxPool2d(2,2)
    self.relu = nn.ReLU()

    self.flatten = nn.Flatten()
    self.fc = nn.Linear(128*12*12,256)
    self.bn_fc = nn.BatchNorm1d(256)

    self.output = nn.Linear(256,7)

  def forward(self,x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.pool(x)


    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu(x)
    x = self.pool(x)

    x = self.conv3(x)
    x = self.bn3(x)
    x = self.relu(x)

    x = self.flatten(x)
    x = self.fc(x)
    x = self.relu(x)
    x = self.output(x)

    return x

model = NetWithBatchNormv2().to(device)



In [68]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=2e-3)

train_loader = DataLoader(train_dataset,batch_size=4*BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4*BATCH_SIZE,shuffle=True)

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [63]:
sanity_check(model,train_loader,"Sanity check CustomCNN with BatchNorm1",2e-3,device,optimizer,criterion)

epoch,▁▂▂▃▄▅▅▆▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁
train/acc,▃▃▁▅▅▃█▆▄▄▁
train/loss,▃▅▄▆▁▃█▃▂▄█
val/acc,▄▆▅▇▇█▆▁▅▆▆
val/loss,▃▄▄▃▄▂▅█▂▁▂
epoch,10
lr,0.0001
train/acc,13.9548
train/loss,2.02876
val/acc,13.02682


Loss @ init: 1.9544 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁███████████████████████████████████████
overfit/loss,█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇█████
init_check/expected_loss,1.94591
init_check/loss,1.95442
init_check/prob_sum,1
init_check/sample_prob,0.13626


In [ ]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="CustomCNN v1 added batchnorm1",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": 2e-3,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 50,
            "batch size" : 256,
        }, checkpoint_path="best_model.pt")

#### Tweaking model to get better result


In [76]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)

In [72]:
train_dataset = CustomDataset(train_df,transform=None)
val_dataset = CustomDataset(val_df,transform=None)

train_loader = DataLoader(train_dataset,batch_size=4*BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4*BATCH_SIZE,shuffle=True)

#### Adding dropout for more regularization ( earlier one was overfitting )

In [82]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout2d = nn.Dropout2d(0.2)
        self.dropout = nn.Dropout(0.5)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(128*12*12, 256)
        self.output = nn.Linear(256, 7)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout2d(x)

        x = self.relu(self.bn3(self.conv3(x)))

        x = self.flatten(x)
        x = self.relu(self.fc(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = Net().to(device)

In [74]:
sanity_check(model,train_loader,"CustomCNN + BatchNorm + Dropout",1e-3,device,optimizer,criterion)

epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇█████
train/loss,█▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁
val/acc,▁▃▃▅▆▇▆▆█▆█▇█▇▇▇▇█████
val/loss,▃▃▂▂▁▁▂▂▁▂▁▂▃▄▄▄▅▅▆▆▇█
epoch,21
lr,0.002
train/acc,98.38464
train/loss,0.0727
val/acc,54.45838


Loss @ init: 1.9473 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁███████████████████████████████████████
overfit/loss,█▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇███
init_check/expected_loss,1.94591
init_check/loss,1.94729
init_check/prob_sum,1
init_check/sample_prob,0.14122


In [79]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="CustomCNN + BatchNorm + Dropout Train",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": 1e-3,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 50,
            "batch size" : 256,
        }, checkpoint_path="best_model.pt")

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇███
train/loss,█▇▇▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁
val/acc,▁▂▃▃▅▅▅▆▆▆▆▆▆▆▇▆▇▇▇▆▇▇▆▇▇▇▇▇▇▇▇▇████████
val/loss,█▇▆▅▄▅▃▃▃▂▃▃▂▂▂▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁
epoch,49
lr,0.001
train/acc,44.36365
train/loss,1.32407
val/acc,50.53988


#### Added scheduler to change learning rate during training 

In [81]:
torch.manual_seed(42) 

model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=80, run_name="CustomCNN + BatchNorm + Dropout + Scheduler",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 80,
            "scheduler": "ReduceLROnPlateau",
        }, checkpoint_path="best_model_v2.pt")

epoch,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇████
lr,███████████████▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇███████████████
train/loss,█▇▇▇▇▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▃▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██▇██████████████████
val/loss,█▆▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▂▁▁▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂
epoch,79
lr,0.0
train/acc,65.65942
train/loss,0.84032
val/acc,57.74991


#### Augmenting Train Data to improve performance 

In [83]:
torch.manual_seed(42) 

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

train_dataset = CustomDataset(train_df,transform=train_transform)
val_dataset = CustomDataset(val_df,transform=None)

train_loader = DataLoader(train_dataset,batch_size=4*BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4*BATCH_SIZE,shuffle=True)


model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=80, run_name="CustomCNN + BatchNorm + Dropout + Scheduler + Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 80,
            "scheduler": "ReduceLROnPlateau",
        }, checkpoint_path="best_model_v3.pt")

epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█████
train/loss,█▇▇▇▇▆▅▅▅▅▅▅▅▅▅▄▅▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁
val/acc,▁▁▂▃▃▃▃▄▄▃▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇█▇█████
val/loss,█▇▆▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch,79
lr,0.001
train/acc,56.48539
train/loss,1.1478
val/acc,60.97179


In [14]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        # Parse all pixel strings once, upfront
        pixels = np.stack([
            np.array(p.split(), dtype=np.float32).reshape(1, 48, 48) / 255.0
            for p in df['pixels'].values
        ])
        self.pixels = torch.tensor(pixels)  # shape: (N, 1, 48, 48)
        self.labels = df['emotion'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.pixels[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [16]:
torch.manual_seed(42) 

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

train_dataset = CustomDataset(train_df,transform=train_transform)
val_dataset = CustomDataset(val_df,transform=None)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, persistent_workers=True)

In [ ]:


train_dataset = CustomDataset(train_df,transform=train_transform)
val_dataset = CustomDataset(val_df,transform=None)

train_loader = DataLoader(train_dataset,batch_size=4*BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4*BATCH_SIZE,shuffle=True)


model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=80, run_name="CustomCNN + BatchNorm + Dropout + Scheduler + Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 80,
            "scheduler": "ReduceLROnPlateau",
        }, checkpoint_path="best_model_v3.pt")

#### Adding 4th Conv Layer to the model
* Using same config as previous run ( Augmentation + Dropout + Batchnorm + Scheduler + Augmentation)


In [25]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout2d = nn.Dropout2d(0.2)
        self.dropout = nn.Dropout(0.5)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, 256)
        self.output = nn.Linear(256, 7)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)                          # (32, 24, 24)
        x = self.dropout2d(x)

        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)                          # (64, 12, 12)
        x = self.dropout2d(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)                          # (128, 6, 6)
        x = self.dropout2d(x)

        x = self.relu(self.bn4(self.conv4(x)))    # (256, 6, 6)

        x = self.flatten(x)                       # 256*6*6 = 9216
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = Net().to(device)

In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)



In [22]:
train_loader = DataLoader(train_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, persistent_workers=True)

In [23]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=80, run_name="CustomCNN4 + BatchNorm + Dropout + Scheduler + Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 150,
            "scheduler": "ReduceLROnPlateau",
        }, checkpoint_path="best_model_v3.pt")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ldavi22 (ldavi22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▃▄▄▄▄▄▄▄▄▄▅▅▅▅▆▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇██████
train/loss,█▇▆▆▆▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁▃▄▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████████
val/loss,█▇▆▆▅▅▄▄▄▄▃▃▃▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁
epoch,79
lr,0.001
train/acc,63.29081
train/loss,0.99799
val/acc,63.04424


In [38]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="run",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra=None, checkpoint_path="best_model.pt",
                 augment_fn=None, device=None):

    config = {"epochs": epochs}
    if config_extra:
        config.update(config_extra)

    run = wandb.init(entity=entity, project=project, name=run_name, config=config)
    wandb.watch(model, criterion, log="all", log_freq=10)

    n_train_samples = len(train_loader.dataset)
    n_val_samples = len(val_loader.dataset)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss_train = 0
        total_acc_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            if augment_fn is not None:
                inputs = augment_fn(inputs)

            output = model(inputs)
            loss = criterion(output, labels)
            total_loss_train += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_acc_train += (torch.argmax(output, axis=1) == labels).sum().item()

        model.eval()
        total_loss_val = 0
        total_acc_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                output = model(inputs)
                loss = criterion(output, labels)
                total_loss_val += loss.item()
                total_acc_val += (torch.argmax(output, axis=1) == labels).sum().item()

        avg_train_loss = total_loss_train / len(train_loader)
        avg_val_loss = total_loss_val / len(val_loader)
        avg_train_acc = (total_acc_train / n_train_samples) * 100
        avg_val_acc = (total_acc_val / n_val_samples) * 100

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        if scheduler is not None:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": avg_train_acc,
            "val/loss": avg_val_loss,
            "val/acc": avg_val_acc,
            "lr": optimizer.param_groups[0]["lr"],
            "epoch": epoch
        })

    wandb.finish()
    return history

In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)



In [34]:
train_dataset = CustomDataset(train_df, transform=None)
val_dataset = CustomDataset(val_df, transform=None)

In [35]:
train_loader = DataLoader(train_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True, persistent_workers=True)

In [36]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])


In [39]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=150, run_name="CustomCNN4 + BatchNorm + Dropout + Scheduler + Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 150,
            "scheduler": "ReduceLROnPlateau",
        }, checkpoint_path="best_model_v3.pt",augment_fn=train_transform,device=device)

epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
lr,█████▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▁▁▁▁
train/acc,▁▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train/loss,█▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▂▃▄▄▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█▇▇█▇██████████
val/loss,█▆▆▆▅▅▅▅▄▄▄▃▄▃▃▃▃▃▃▃▃▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,149
lr,3e-05
train/acc,69.10785
train/loss,0.83522
val/acc,65.18635
